# Условия:
N = 2 - сколько типов процессорв

m_1 = 3 - сколько процов 1 типа

m_2 = 1 - сколько процов 2 типа

n = 2 - сколько шин, спсобов передачи даннных

T_о1 = 1.12 - время выполнеия операции процом 1 типа

T_02 = 6.2

Tau_1 = 1.33 - интенсивность обслуживания для 1 проца

Tau_2 = 1.5

q = 0.35 - параметр связанности алгоритма по памяти

In [92]:
N = 2
m1, m2 = 3, 1
n = 2
To1, To2 = 1.12, 6.2
tau1, tau2 = 1.33, 1.5
q = 0.35

In [93]:
v1 = (q/n) / (To1 - (q/n)*tau1)
v2 = (q/n) / (To2 - (q/n)*tau2)
mu1 = 1/tau1
mu2 = 1/tau2

rho1 = v1/mu1
rho2 = v2/mu2

display(Markdown(rf"* $\nu_1 = {v1}$"))
display(Markdown(rf"* $\nu_2 = {v2}$"))
display(Markdown(rf"* $\mu_1 = {mu1}$"))
display(Markdown(rf"* $\mu_2 = {mu2}$"))

* $\nu_1 = 0.1972386587771203$

* $\nu_2 = 0.029473684210526315$

* $\mu_1 = 0.7518796992481203$

* $\mu_2 = 0.6666666666666666$

Условие стационарного режима:

$\forall i: \rho_i = \frac{\nu_i}{\mu_i} \le 1$

In [94]:
ro1 = v1/mu1
ro2 = v2/mu2
display(Markdown(rf"* $\rho_1 = {ro1} \le 1$"))
display(Markdown(rf"* $\rho_2 = {ro2} \le 1$"))

* $\rho_1 = 0.26232741617357 \le 1$

* $\rho_2 = 0.04421052631578948 \le 1$

# Решение СЛАУ и поиск вероятностей состояний

In [95]:
A = [
      [-(3*v1+v2),     mu1,                  mu2,                  0,                    0,                    0,                                  0,                    0],
      [3*v1,           -(mu1+2*v1+v2),       0,                    mu2,                  2*mu1,                0,                                  0,                    0],
      [v2,             0,                    -(mu2+3*v1),          mu1,                  0,                    0,                                  0,                    0],
      [0,              v2,                   3*v1,                 -(mu1+mu2+2*v1),      0,                    4*mu1/3,                            0,                    0],
      [0,              2*v1,                 0,                    0,                    -(2*mu1+v1+v2),       2*mu2/3,                            2*mu1,                0],
      [0,              0,                    0,                    2*v1,                 v2,                   -(4*mu1/3+2*mu2/3+v1),              0,                    3*mu1/2],
      [0,              0,                    0,                    0,                    v1,                   0,                                  -(2*mu1+v2),          mu2/2],
      [0,              0,                    0,                    0,                    0,                    v1,                                 v2,                   -(3*mu1/2+mu2/2)]
     ]

In [96]:
A = np.array(A)
np.set_printoptions(linewidth=150)
print(A)

[[-0.62118966  0.7518797   0.66666667  0.          0.          0.          0.          0.        ]
 [ 0.59171598 -1.1758307   0.          0.66666667  1.5037594   0.          0.          0.        ]
 [ 0.02947368  0.         -1.25838264  0.7518797   0.          0.          0.          0.        ]
 [ 0.          0.02947368  0.59171598 -1.81302368  0.          1.00250627  0.          0.        ]
 [ 0.          0.39447732  0.          0.         -1.73047174  0.44444444  1.5037594   0.        ]
 [ 0.          0.          0.          0.39447732  0.02947368 -1.64418937  0.          1.12781955]
 [ 0.          0.          0.          0.          0.19723866  0.         -1.53323308  0.33333333]
 [ 0.          0.          0.          0.          0.          0.19723866  0.02947368 -1.46115288]]


Одно из уравнений нужно заменить на условие нормировки.

$P_{00}^{00}+P_{10}^{10}+P_{01}^{10}+P_{11}^{20}+P_{20}^{20}+P_{21}^{21}+P_{30}^{21}+P_{31}^{22} = 1$

In [97]:
eq_num = 3
A[eq_num] = [1,1,1,1,1,1,1,1]
print(A)

[[-0.62118966  0.7518797   0.66666667  0.          0.          0.          0.          0.        ]
 [ 0.59171598 -1.1758307   0.          0.66666667  1.5037594   0.          0.          0.        ]
 [ 0.02947368  0.         -1.25838264  0.7518797   0.          0.          0.          0.        ]
 [ 1.          1.          1.          1.          1.          1.          1.          1.        ]
 [ 0.          0.39447732  0.          0.         -1.73047174  0.44444444  1.5037594   0.        ]
 [ 0.          0.          0.          0.39447732  0.02947368 -1.64418937  0.          1.12781955]
 [ 0.          0.          0.          0.          0.19723866  0.         -1.53323308  0.33333333]
 [ 0.          0.          0.          0.          0.          0.19723866  0.02947368 -1.46115288]]


In [98]:
B = np.array([0,0,0,0,0,0,0,0])
B[eq_num] = 1
print("B =",B)

B = [0 0 0 1 0 0 0 0]


In [99]:
P = np.linalg.solve(A,B)
P_res = 0
P_names = ['$P_{00}^{00}$', '$P_{10}^{10}$',
  '$P_{01}^{10}$', '$P_{11}^{20}$', '$P_{20}^{20}$',
  '$P_{21}^{21}$', '$P_{30}^{21}$', '$P_{31}^{22}$']
for i in range(8):
  P_res = P_res + P[i]
  display(Markdown(rf"{P_names[i]} = %.6f"%P[i]))
display(P_res)

$P_{00}^{00}$ = 0.472680

$P_{10}^{10}$ = 0.371991

$P_{01}^{10}$ = 0.020897

$P_{11}^{20}$ = 0.016446

$P_{20}^{20}$ = 0.097583

$P_{21}^{21}$ = 0.006471

$P_{30}^{21}$ = 0.012799

$P_{31}^{22}$ = 0.001132

1.0

# Оценка производительности

Совокупные вероятности:

$P_{00} = P_{00}^{00}$

$P_{10} = P_{10}^{10} + P_{01}^{10}$

$P_{20} = P_{20}^{20} + P_{11}^{20}$

$P_{21} = P_{30}^{21} + P_{21}^{21}$

$P_{22} = P_{31}^{22}$

Средняя длина очереди:

$l_{ср} = \sum{l \cdot P_{l,k}}$

$l_{ср} = 0 \cdot P_{0,0} + 0 \cdot P_{1,0} + 0 \cdot P_{2,0} + 1 \cdot P_{2,1} + 2 \cdot P_{2,2}$

In [100]:
P00 = P[0]
P10 = P[1]+P[2]
P20 = P[3]+P[4]
P21 = P[5]+P[6]
P22 = P[7]
l = 0*P00 + 0*P10 + 0*P20 + 1*P21 + 2*P22
display(Markdown(r"$l_{ср}$ = %.6f"%l))

$l_{ср}$ = 0.021534

Относительная потеря производительности:

$ \theta_i = 1 + \frac{l_{ср}\tau_i}{T_{oi}+\tau_i}$

In [101]:
theta1 = 1 + (l*tau1)/(To1+tau1)
theta2 = 1 + (l*tau2)/(To2+tau2)
display(Markdown(r"$\theta_1$ = %.8f"%theta1))
display(Markdown(r"$\theta_2$ = %.8f"%theta2))

$\theta_1$ = 1.01168999

$\theta_2$ = 1.00419497

Производительность:

$ П_{ср} = \sum{\frac{m_i}{\theta_i}}$

In [102]:
Performance = m1/theta1 + m2/theta2
display(Markdown(r"$П_{ср}$ = %.5f"%Performance))

$П_{ср}$ = 3.96116